<a href="https://colab.research.google.com/github/dempsesn/course-workspace/blob/main/Dempsey_Mini_Project_Build_and_Audit_an_MCP_Server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install mcp packaging pandas tabulate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [mcp]


In [2]:
# In project_mcp_server.py
import re
from packaging import version
from mcp.server.mcpserver import MCPServer

# Initialize the MCPServer instance (mcp 2.x standard)
mcp = MCPServer("Version-Aware CVE Lookup")

# Placeholder for KB_ENTRIES to prevent future NameErrors
KB_ENTRIES = [
    {
        "product": "example-lib",
        "fixed_in": "2.0.0",
        "cve_id": "CVE-2023-XXXX",
        "summary": "An example vulnerability patched in 2.0.0"
    }
]

@mcp.tool()
def lookup_cve_version_aware(product: str, current_version: str) -> str:
    """Look up CVEs for a specific software version against the knowledge base."""
    # Read the KB entries and parse vulnerable version thresholds
    # Example logic matching version bounds:
    matches = []
    for entry in KB_ENTRIES:
        if entry["product"].lower() == product.lower():
            # Check if current_version is strictly less than patched_version
            if version.parse(current_version) < version.parse(entry["fixed_in"]):
                matches.append(f"{entry['cve_id']}: {entry['summary']}")
    return "\n".join(matches) if matches else "NO_VULNERABILITIES_FOUND"

In [3]:
# In project_mcp_server.py
@mcp.resource("cve://knowledge-base")
def get_cve_knowledge_base() -> str:
    """Exposes the local markdown knowledge base of CVEs."""
    with open("project_cve_kb.md", "r", encoding="utf-8") as f:
        return f.read()

In [4]:
# Write the local Markdown CVE Knowledge Base file
cve_kb_content = """# CVE Knowledge Base

## CVE-2023-XXXX
- **Product**: example-lib
- **Fixed In**: 2.0.0
- **Summary**: An example vulnerability patched in 2.0.0
"""

with open("project_cve_kb.md", "w", encoding="utf-8") as f:
    f.write(cve_kb_content)

# Combine the previously declared server cells and write to project_mcp_server.py
server_code = """import re
from packaging import version
from mcp.server.mcpserver import MCPServer

mcp = MCPServer("Version-Aware CVE Lookup")

KB_ENTRIES = [
    {
        "product": "example-lib",
        "fixed_in": "2.0.0",
        "cve_id": "CVE-2023-XXXX",
        "summary": "An example vulnerability patched in 2.0.0"
    }
]

@mcp.tool()
def lookup_cve_version_aware(product: str, current_version: str) -> str:
    \"\"\"Look up CVEs for a specific software version against the knowledge base.\"\"\"
    matches = []
    for entry in KB_ENTRIES:
        if entry["product"].lower() == product.lower():
            if version.parse(current_version) < version.parse(entry["fixed_in"]):
                matches.append(f\"{entry['cve_id']}: {entry['summary']}\")
    return "\\n".join(matches) if matches else "NO_VULNERABILITIES_FOUND"

@mcp.resource("cve://knowledge-base")
def get_cve_knowledge_base() -> str:
    \"\"\"Exposes the local markdown knowledge base of CVEs.\"\"\"
    with open("project_cve_kb.md", "r", encoding="utf-8") as f:
        return f.read()

if __name__ == '__main__':
    # Start the server on stdio
    mcp.run()
"""

with open("project_mcp_server.py", "w", encoding="utf-8") as f:
    f.write(server_code)

print("Successfully created project_cve_kb.md and project_mcp_server.py!")

Successfully created project_cve_kb.md and project_mcp_server.py!


In [5]:
import sys
import traceback
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Using sys.executable guarantees the server runs inside the same environment as the notebook
server_params = StdioServerParameters(
    command=sys.executable,
    args=["project_mcp_server.py"],
    env=None
)

try:
    # We redirect stderr logging of the stdio client to sys.__stderr__
    # because Colab's default sys.stderr lacks a valid file descriptor (fileno)
    async with stdio_client(server_params, errlog=sys.__stderr__) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools_list = await session.list_tools()
            for tool in tools_list.tools:
                print(f"Discovered: {tool.name} -> {tool.description}")
except ExceptionGroup as eg:
    print("Captured ExceptionGroup:")
    for exc in eg.exceptions:
        traceback.print_exception(type(exc), exc, exc.__traceback__)
except Exception as e:
    traceback.print_exc()

Discovered: lookup_cve_version_aware -> Look up CVEs for a specific software version against the knowledge base.


In [6]:
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(
    command=sys.executable,
    args=["project_mcp_server.py"],
    env=None
)

# Open the server, initialize the session, and call the tool in a single block
async with stdio_client(server_params, errlog=sys.__stderr__) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        result = await session.call_tool(
            "lookup_cve_version_aware",
            arguments={"product": "example-lib", "current_version": "1.5.0"}
        )
        print("Tool response:", result.content[0].text)

Tool response: CVE-2023-XXXX: An example vulnerability patched in 2.0.0


In [7]:
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(
    command=sys.executable,
    args=["project_mcp_server.py"],
    env=None
)

# Open the server connection, initialize, and read the resource inside a single context
async with stdio_client(server_params, errlog=sys.__stderr__) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        kb_resource = await session.read_resource("cve://knowledge-base")
        print("KB Content preview:\n", kb_resource.contents[0].text[:200])

KB Content preview:
 # CVE Knowledge Base

## CVE-2023-XXXX
- **Product**: example-lib
- **Fixed In**: 2.0.0
- **Summary**: An example vulnerability patched in 2.0.0



In [8]:
_SIMULATED_HOSTS = {
    # Existing test hosts ...
    "192.168.1.50": [
        {"port": 22, "service": "ssh", "banner": "OpenSSH_7.4p1 Debian-10"},
        {"port": 80, "service": "http", "banner": "Apache/2.4.52 (Ubuntu)"},
        {"port": 6379, "service": "redis", "banner": "Redis key-value store 7.0.11"}
    ]
}

In [9]:
@mcp.tool()
def check_port_policy(port: int, environment: str = "production") -> str:
    """Evaluate whether an open network port conforms to security baseline policies.

    Args:
        port: The TCP/UDP port number to validate.
        environment: Deployment environment ('production', 'staging', or 'internal').
    Returns:
        JSON string indicating 'status' (COMPLIANT or VIOLATION) and policy remediation steps.
    """
    RESTRICTED_PORTS = {
        "production": [21, 23, 3389, 6379],  # Telnet, FTP, RDP, unauthenticated DB
        "staging": [23]
    }
    restricted = RESTRICTED_PORTS.get(environment.lower(), [])
    if port in restricted:
        return f'{{"port": {port}, "status": "VIOLATION", "action": "Block port {port} at edge firewall."}}'
    return f'{{"port": {port}, "status": "COMPLIANT", "action": "Port {port} permitted."}}'

In [10]:
from pathlib import Path

# In a notebook cell:
BASE_DIR = Path.cwd()
KB_PATH = BASE_DIR / "project_cve_kb.md"
EXPECTED_PATH = BASE_DIR / "expected_cves.json"

In [11]:
expected_cves = {
  "192.168.1.50": {
    "22": {
      "expected_cve": "CVE-2021-28041",
      "state": "grounded"
    },
    "80": {
      "expected_cve": None,
      "state": "blind_spot"
    },
    "6379": {
      "expected_cve": None,
      "state": "true_gap"
    }
  }
}

In [12]:
%%writefile project_cve_kb.md
## OpenSSH 7.4p1
- Product: OpenSSH
- Fixed In: 8.5p1
- CVE: CVE-2021-28041
- Summary: Integer overflow in ssh-agent forwarder leading to memory corruption.

## Apache 2.4.49
- Product: Apache
- Fixed In: 2.4.51
- CVE: CVE-2021-41773
- Summary: Path traversal and file disclosure flaw in Apache HTTP Server.

Overwriting project_cve_kb.md


In [13]:
csv_data = """host,port,service,banner,tool_used,result_status,cve_reported,audit_category
192.168.1.50,22,ssh,OpenSSH_7.4p1,lookup_cve_naive,FLAGGED,CVE-2021-28041,grounded
192.168.1.50,22,ssh,OpenSSH_7.4p1,lookup_cve_version_aware,FLAGGED,CVE-2021-28041,grounded
192.168.1.50,80,http,Apache/2.4.52,lookup_cve_naive,FLAGGED,CVE-2021-41773,wrong_version
192.168.1.50,80,http,Apache/2.4.52,lookup_cve_version_aware,CLEAN,NONE,grounded
192.168.1.50,6379,redis,Redis 7.0.11,lookup_cve_version_aware,CLEAN,NONE,true_gap"""